# Fine-tune a font-similarity image encoder

Fine-tunes the SigLIP (`ViT-B-16-SigLIP` / `webli`) image encoder typeseek already uses, but swaps its training objective. The pretrained checkpoint was trained to match images to captions -- a different task from ours. Here we use ArcFace-style metric learning (the same technique face-recognition models use) so that renders of the *same* font cluster tightly in embedding space and different fonts separate clearly, regardless of weight, background, or capture noise.

Runs locally (Apple Silicon via PyTorch's MPS backend, or CUDA/CPU elsewhere) -- no Colab or Drive needed.

**Before running:**
1. Launch Jupyter from the repo root so relative paths resolve correctly.
2. `data/training/` (generated by `ingestion/generate_training_data.py`) must exist locally -- copy it over from wherever it was generated if this is a new machine, since it's gitignored and not part of the repo itself.
3. Real photos of known fonts go in `data/real_photos/<font-slug>/*.jpg` -- the evaluation cells pick them up automatically if present.

In [1]:
%pip install -q open_clip_torch

import torch

if torch.cuda.is_available():
    print("device: cuda --", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    print("device: mps (Apple Silicon)")
else:
    print("device: cpu")

Note: you may need to restart the kernel to use updated packages.
device: mps (Apple Silicon)


In [2]:
from pathlib import Path
import os

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

PROJECT_ROOT = REPO_ROOT / "data/finetune"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
TRAIN_ROOT = REPO_ROOT / "data/training"
REAL_PHOTOS_ROOT = REPO_ROOT / "data/real_photos"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

assert TRAIN_ROOT.exists(), f"{TRAIN_ROOT} not found -- copy it over from wherever it was generated"
print("repo root:", REPO_ROOT)
print("train root contents (sample):", os.listdir(TRAIN_ROOT)[:5])

repo root: /Users/dickson/Code/typeseek
train root contents (sample): ['google-fonts-cute-font', 'google-fonts-andika', 'google-fonts-playwrite-nz-guides', 'google-fonts-gelasio', 'google-fonts-gidole']


## Config

In [3]:
MODEL_NAME = "ViT-B-16-SigLIP"
PRETRAINED = "webli"
EMBED_DIM = 768

BATCH_SIZE = 256
EPOCHS = 15
PROJECTION_LR = 1e-4
HEAD_LR = 3e-4
ARCFACE_SCALE = 30.0
ARCFACE_MARGIN = 0.3
VAL_FRACTION = 0.1
# Notebook-defined Dataset classes cannot be spawned reliably from __main__.
NUM_WORKERS = 0
CACHE_BATCH_SIZE = 64
EMBED_CACHE_DIR = PROJECT_ROOT / "embedding_cache"
os.makedirs(EMBED_CACHE_DIR, exist_ok=True)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# bf16 autocast is CUDA-only here; MPS/CPU run in full precision for correctness
AMP_ENABLED = device == "cuda"
print(f"device: {device}; embedding cache: {EMBED_CACHE_DIR}")

device: mps; embedding cache: /Users/dickson/Code/typeseek/data/finetune/embedding_cache


## Dataset -- stratified per-class train/val split

A random overall split can leave low-sample classes entirely out of validation; splitting per class avoids that.

In [4]:
import random
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T


def build_stratified_split(root, val_fraction=0.1, seed=42):
    root = Path(root)
    classes = sorted(d.name for d in root.iterdir() if d.is_dir())
    class_to_idx = {c: i for i, c in enumerate(classes)}
    rng = random.Random(seed)
    train_samples, val_samples = [], []
    samples_by_class = {}
    for c in classes:
        files = sorted((root / c).glob("*.jpg"))
        rng.shuffle(files)
        samples_by_class[c] = [str(f) for f in files]
        n_val = max(1, int(len(files) * val_fraction)) if len(files) > 3 else 0
        val_files, train_files = files[:n_val], files[n_val:]
        label = class_to_idx[c]
        train_samples += [(str(f), label) for f in train_files]
        val_samples += [(str(f), label) for f in val_files]
    return classes, class_to_idx, train_samples, val_samples, samples_by_class


# fresh randomness per __getitem__ call -- prevents memorizing the fixed pre-baked images
LIVE_AUGMENT = T.Compose([
    T.RandomAffine(degrees=10, translate=(0.08, 0.08), fill=255),
    T.RandomPerspective(distortion_scale=0.2, p=0.4, fill=255),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05),
    T.RandomApply([T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.5))], p=0.5),
    T.RandomGrayscale(p=0.15),
    T.RandomResizedCrop(size=(256, 512), scale=(0.7, 1.0), ratio=(1.7, 2.3)),
])


class FontDataset(Dataset):
    def __init__(self, samples, transform, augment=False):
        self.samples = samples
        self.transform = transform
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.augment:
            image = LIVE_AUGMENT(image)
        return self.transform(image), label


classes, class_to_idx, train_samples, val_samples, samples_by_class = build_stratified_split(
    TRAIN_ROOT, VAL_FRACTION
)
print(f"{len(classes)} classes, {len(train_samples)} train images, {len(val_samples)} val images")

1910 classes, 128480 train images, 13312 val images


## Model -- SigLIP image encoder + ArcFace head

In [6]:
import contextlib

import open_clip
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=PRETRAINED)
model = model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False


def encode_frozen(images):
    amp = torch.autocast(device_type="cuda", dtype=torch.bfloat16) if AMP_ENABLED else contextlib.nullcontext()
    with torch.no_grad(), amp:
        return model.encode_image(images).float()


def cache_embeddings(samples, cache_path):
    if cache_path.exists():
        cached = torch.load(cache_path, map_location="cpu")
        print(f"loaded {len(cached['labels'])} cached embeddings from {cache_path.name}")
        return cached["embeddings"], cached["labels"]

    loader = DataLoader(
        FontDataset(samples, preprocess),
        batch_size=CACHE_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda"),
    )
    embeddings, labels = [], []
    for images, batch_labels in tqdm(loader, desc=f"caching {cache_path.stem}"):
        embeddings.append(encode_frozen(images.to(device, non_blocking=True)).cpu())
        labels.append(batch_labels)
    cached_embeddings = torch.cat(embeddings)
    cached_labels = torch.cat(labels)
    torch.save({"embeddings": cached_embeddings, "labels": cached_labels}, cache_path)
    print(f"saved {len(cached_labels)} embeddings to {cache_path}")
    return cached_embeddings, cached_labels


train_embeddings, train_labels = cache_embeddings(train_samples, EMBED_CACHE_DIR / "train.pt")
val_embeddings, val_labels = cache_embeddings(val_samples, EMBED_CACHE_DIR / "val.pt")

train_loader = DataLoader(
    torch.utils.data.TensorDataset(train_embeddings, train_labels),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
)
val_loader = DataLoader(
    torch.utils.data.TensorDataset(val_embeddings, val_labels),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
)


class ProjectionHead(nn.Module):
    def __init__(self, dim, hidden_dim=512, dropout=0.1):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, dim),
        )
        self.block2 = nn.Sequential(
            nn.LayerNorm(dim), nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        x = x + self.block1(x)
        x = x + self.block2(x)
        return x


class ArcFaceHead(nn.Module):
    def __init__(self, embedding_dim, num_classes, scale=30.0, margin=0.3):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.scale = scale
        self.margin = margin

    def forward(self, embeddings, labels=None):
        embeddings = F.normalize(embeddings.float(), dim=-1)
        weight = F.normalize(self.weight, dim=-1)
        cosine = embeddings @ weight.t()
        if labels is None:
            return cosine * self.scale
        theta = torch.acos(cosine.clamp(-1 + 1e-7, 1 - 1e-7))
        target_logits = torch.cos(theta + self.margin)
        one_hot = F.one_hot(labels, num_classes=weight.shape[0]).float()
        logits = one_hot * target_logits + (1 - one_hot) * cosine
        return logits * self.scale


projection_head = ProjectionHead(EMBED_DIM).to(device)
arcface = ArcFaceHead(EMBED_DIM, len(classes), ARCFACE_SCALE, ARCFACE_MARGIN).to(device)

caching train: 100%|██████████| 2008/2008 [29:10<00:00,  1.15it/s]


saved 128480 embeddings to /Users/dickson/Code/typeseek/data/finetune/embedding_cache/train.pt


caching val: 100%|██████████| 208/208 [03:06<00:00,  1.11it/s]

saved 13312 embeddings to /Users/dickson/Code/typeseek/data/finetune/embedding_cache/val.pt


## Training loop -- resumable head training from cached SigLIP embeddings

The frozen SigLIP backbone is evaluated once in the model cell. This loop trains only the projection and ArcFace heads, so it should be fast and low-heat on a local Mac.

In [7]:
import torch.optim as optim
from tqdm.auto import tqdm

optimizer = optim.AdamW([
    {"params": projection_head.parameters(), "lr": PROJECTION_LR},
    {"params": arcface.parameters(), "lr": HEAD_LR},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))

ckpt_path = CHECKPOINT_DIR / "latest.pt"
start_epoch = 0
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    projection_head.load_state_dict(ckpt["projection_head"])
    arcface.load_state_dict(ckpt["arcface"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    print(f"resumed from epoch {start_epoch}")

for epoch in range(start_epoch, EPOCHS):
    projection_head.train()
    arcface.train()
    total_loss = 0.0
    for base_embeddings, labels in tqdm(train_loader, desc=f"epoch {epoch}"):
        base_embeddings = base_embeddings.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        embeddings = projection_head(base_embeddings)
        logits = arcface(embeddings, labels)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"epoch {epoch}: train_loss={total_loss / len(train_loader):.4f}")
    torch.save({
        "epoch": epoch,
        "projection_head": projection_head.state_dict(),
        "arcface": arcface.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
    }, ckpt_path)
    torch.save(
        {"epoch": epoch, "projection_head": projection_head.state_dict(), "arcface": arcface.state_dict()},
        CHECKPOINT_DIR / f"epoch_{epoch}.pt",
    )

epoch 0: 100%|██████████| 501/501 [00:03<00:00, 131.85it/s]


epoch 0: train_loss=15.2196


epoch 1: 100%|██████████| 501/501 [00:03<00:00, 146.98it/s]


epoch 1: train_loss=13.3564


epoch 2: 100%|██████████| 501/501 [00:03<00:00, 144.14it/s]


...

epoch 12: train_loss=8.5260


epoch 13: 100%|██████████| 501/501 [00:03<00:00, 149.42it/s]


epoch 13: train_loss=8.4802


epoch 14: 100%|██████████| 501/501 [00:03<00:00, 149.42it/s]

epoch 14: train_loss=8.4577


## Evaluation -- rank of the true font among ALL classes

This is the metric that actually matters (matches what the product does): build a per-class gallery embedding, then for each query check where the true class ranks among *every* class -- not just top-1/top-5 within a random batch.

In [ ]:
@torch.no_grad()
def build_gallery(max_per_class=5):
    projection_head.eval()
    gallery = torch.zeros(len(classes), EMBED_DIM, device=device)
    for c, idx in class_to_idx.items():
        paths = samples_by_class[c][:max_per_class]
        imgs = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths]).to(device)
        embs = projection_head(encode_frozen(imgs))
        gallery[idx] = F.normalize(embs.float(), dim=-1).mean(dim=0)
    return F.normalize(gallery, dim=-1)


@torch.no_grad()
def rank_query(gallery, image_path, true_class_idx):
    projection_head.eval()
    img = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)
    emb = projection_head(encode_frozen(img))
    emb = F.normalize(emb.float(), dim=-1)
    sims = (gallery @ emb.t()).squeeze(1)
    order = sims.argsort(descending=True)
    rank = (order == true_class_idx).nonzero(as_tuple=True)[0].item() + 1
    return rank, sims[true_class_idx].item()


gallery = build_gallery()

# synthetic held-out val images, as a proxy metric
ranks = [rank_query(gallery, path, label)[0] for path, label in random.sample(val_samples, min(200, len(val_samples)))]
print(f"synthetic val: median rank={sorted(ranks)[len(ranks)//2]}, top-1={sum(r==1 for r in ranks)/len(ranks):.2%}, top-5={sum(r<=5 for r in ranks)/len(ranks):.2%}")

In [ ]:
# real photos of known fonts
if os.path.exists(REAL_PHOTOS_ROOT):
    real_root = Path(REAL_PHOTOS_ROOT)
    real_results = []
    for class_dir in sorted(real_root.iterdir()):
        if class_dir.name not in class_to_idx:
            print(f"skipping {class_dir.name}: not in training classes")
            continue
        true_idx = class_to_idx[class_dir.name]
        for photo in class_dir.glob("*.*"):
            rank, sim = rank_query(gallery, str(photo), true_idx)
            real_results.append((class_dir.name, photo.name, rank, sim))
            print(f"{class_dir.name:30s} {photo.name:20s} rank={rank:5d} sim={sim:.4f}")

    if real_results:
        ranks = [r[2] for r in real_results]
        print(f"\nreal photos: median rank={sorted(ranks)[len(ranks)//2]}, "
              f"top-1={sum(r==1 for r in ranks)/len(ranks):.2%}, top-5={sum(r<=5 for r in ranks)/len(ranks):.2%}")
else:
    print(f"no real photos found at {REAL_PHOTOS_ROOT} -- upload real_photos.zip to Drive and re-run the extraction cell")

## Per-epoch real-photo trend

Every epoch's checkpoint is already saved -- check whether real-photo generalization peaked earlier and got overfit away by the final epoch, before redesigning augmentation further.

In [ ]:
import glob

real_root = Path(REAL_PHOTOS_ROOT)
real_samples = [
    (str(photo), class_to_idx[class_dir.name])
    for class_dir in sorted(real_root.iterdir())
    if class_dir.name in class_to_idx
    for photo in class_dir.glob("*.*")
]

checkpoint_paths = sorted(
    glob.glob(f"{CHECKPOINT_DIR}/epoch_*.pt"),
    key=lambda p: int(p.split("epoch_")[-1].split(".pt")[0]),
)

print(f"{'epoch':>6s} {'median_rank':>12s} {'top1':>8s} {'top5':>8s}")
for ckpt_path in checkpoint_paths:
    ckpt = torch.load(ckpt_path, map_location=device)
    projection_head.load_state_dict(ckpt["projection_head"])
    gallery = build_gallery()
    ranks = [rank_query(gallery, path, label)[0] for path, label in real_samples]
    median = sorted(ranks)[len(ranks) // 2]
    top1 = sum(r == 1 for r in ranks) / len(ranks)
    top5 = sum(r <= 5 for r in ranks) / len(ranks)
    print(f"{ckpt['epoch']:6d} {median:12d} {top1:8.2%} {top5:8.2%}")

## Diagnose real-photo results before changing anything else

Three questions, using checkpoints we already have -- no retraining:
1. Is the 5-images-per-class gallery too narrow to be a fair reference?
2. What is the model actually confusing each real photo with (top-5, not just true-class rank)?
3. Which results are stable across checkpoints (real signal) vs volatile (noise)?

In [ ]:
idx_to_class = {v: k for k, v in class_to_idx.items()}

# --- 1. gallery size sensitivity: 5 samples/class vs a much larger sample ---
ckpt = torch.load(f"{CHECKPOINT_DIR}/epoch_4.pt", map_location=device)
projection_head.load_state_dict(ckpt["projection_head"])

gallery_small = build_gallery(max_per_class=5)
gallery_large = build_gallery(max_per_class=40)

print(f"{'photo':30s} {'rank(gallery=5)':>16s} {'rank(gallery=40)':>17s}")
for photo_path, true_idx in real_samples:
    r_small, _ = rank_query(gallery_small, photo_path, true_idx)
    r_large, _ = rank_query(gallery_large, photo_path, true_idx)
    name = Path(photo_path).parent.name.replace("google-fonts-", "") + "/" + Path(photo_path).name
    print(f"{name:30s} {r_small:16d} {r_large:17d}")

In [ ]:
# --- 2. what is the model actually confusing each photo with? (top-5 predicted classes) ---
print(f"{'photo':30s} {'true_class':22s} top-5 predicted")
for photo_path, true_idx in real_samples:
    img = preprocess(Image.open(photo_path).convert("RGB")).unsqueeze(0).to(device)
    emb = F.normalize(projection_head(encode_frozen(img)), dim=-1)
    sims = (gallery_large @ emb.t()).squeeze(1)
    top5 = sims.argsort(descending=True)[:5].tolist()
    top5_names = [f"{idx_to_class[i].replace('google-fonts-', '')}({sims[i]:.2f})" for i in top5]
    name = Path(photo_path).parent.name.replace("google-fonts-", "") + "/" + Path(photo_path).name
    true_name = idx_to_class[true_idx].replace("google-fonts-", "")
    print(f"{name:30s} {true_name:22s} {', '.join(top5_names)}")

In [ ]:
# --- 3. stability across checkpoints: epoch 4 (best median) vs epoch 14 (final) ---
ckpt4 = torch.load(f"{CHECKPOINT_DIR}/epoch_4.pt", map_location=device)
projection_head.load_state_dict(ckpt4["projection_head"])
gallery4 = build_gallery(max_per_class=40)
ranks4 = {p: rank_query(gallery4, p, l)[0] for p, l in real_samples}

ckpt14 = torch.load(f"{CHECKPOINT_DIR}/epoch_14.pt", map_location=device)
projection_head.load_state_dict(ckpt14["projection_head"])
gallery14 = build_gallery(max_per_class=40)
ranks14 = {p: rank_query(gallery14, p, l)[0] for p, l in real_samples}

print(f"{'photo':30s} {'rank@epoch4':>12s} {'rank@epoch14':>13s} {'delta':>8s}")
for photo_path, _ in real_samples:
    r4, r14 = ranks4[photo_path], ranks14[photo_path]
    name = Path(photo_path).parent.name.replace("google-fonts-", "") + "/" + Path(photo_path).name
    print(f"{name:30s} {r4:12d} {r14:13d} {r14 - r4:+8d}")

## Export the fine-tuned encoder

Only the image encoder is exported -- text search keeps using the original, unmodified SigLIP checkpoint.

In [ ]:
FINAL_EXPORT_PATH = f"{PROJECT_ROOT}/font_encoder_final.pt"
torch.save({
    "projection_head_state_dict": projection_head.state_dict(),
    "model_name": MODEL_NAME,
    "pretrained_base": PRETRAINED,
    "embedding_dim": EMBED_DIM,
    "classes": classes,
}, FINAL_EXPORT_PATH)
print("exported to", FINAL_EXPORT_PATH)